# 01 · 数据准备（Colab）

下载语料、建清单、按平稳性给噪声分类、合成固定测试集。

## 数据设计

| 用途 | 数据集 | 规模 | 为什么是它 |
|---|---|---|---|
| **训练干净语音** | DNS Challenge 4 `read_speech` | 1 分片 ≈ 21 小时 | 学术界公认基准；分片独立可单独解压，能做小规模子集 |
| **训练噪声** | DNS `noise_fullband`（AudioSet + Freesound） | 3 分片 ≈ 13 GB | 真实录制，覆盖面广；按平稳性自动分成两组 |
| **房间冲激响应** | DNS `impulse_responses` + 本项目合成 | 5.9 GB + 0 | 真实的验证泛化，合成的做**受控 RT60 扫描** |
| **中文 ASR 评测** | WenetSpeech `test_meeting` | 220 MB parquet | 真实会议录音，带中文转写 |

## 核心设计：训练用英文，评测用中文

这不是凑合，是**刻意的**。前端增强模型在 DNS（英文为主）上训练，
却直接在中文 WenetSpeech 上测 CER —— 如果 CER 仍然改善，就证明模型学到的是
**纯粹的声学/频域去噪规律**，而不是过拟合到某个语言的发音模式。
同语种自训自测拿不到这个论证。

## 为什么噪声要区分稳态 / 非稳态

这是"传统 DSP vs 神经网络"这条对照的核心抓手：

- **稳态**（风扇、空调、白噪声）：谱统计平稳，MCRA 类噪声估计跟得上，
  谱减/维纳/MMSE-LSA 表现好且算力极低；
- **非稳态**（键盘、关门、餐具、babble）：统计特性瞬变，传统方法跟不上噪声谱，
  失效或产生明显音乐噪声 —— 这正是神经网络该赢的地方。

DNS 的噪声文件**不带平稳性标注**，所以用 `rtse.dsp.stationarity` 从信号本身算
（去趋势帧能量动态范围），本地已在 7 类已知噪声上验证过分类正确。

## 为什么 RIR 要合成 + 真实两种都用

- **合成 RIR**（镜像源法）：参数完全可控，能精确指定 RT60，
  做 0.2 / 0.4 / 0.6 / 0.8 s 的**严格量化扫描**；
- **真实 RIR**（DNS 实测）：含非均匀漫反射、墙面材质不对称吸收、麦克风频响失真，
  检验模型在真实声学环境下的泛化，避免"仿真过拟合"。

两者在测试集里**并行分层**，可以直接对比"合成 RIR 上的成绩比真实 RIR 好多少"。

## 执行前

1. 先跑「配置」cell，确认 `DRIVE_ROOT` 与你的实际目录一致
2. `rtse-colab.zip` 必须已上传到 `DRIVE_ROOT`（本地 `uv run python scripts/pack_for_colab.py` 生成）
3. **第一次建议先把 `QUICK_TEST = True`** 跑一遍（只下 WenetSpeech 约 520 MB，验证链路）

In [ ]:
# ── 挂载 Google Drive ───────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  配置 —— 所有路径与开关集中在这一个 cell，别处不要再写死路径
# ═══════════════════════════════════════════════════════════════════════

# Drive 上的项目根。**持久**，会话结束不丢。
DRIVE_ROOT = '/content/drive/MyDrive/Audio AI/RTSE'

# Colab 本地盘。**临时**，会话结束即消失，但读写比 Drive 快得多。
WORK_ROOT = '/content/rtse_work'

# ── 语料怎么放 ─────────────────────────────────────────────────────────
#   'hybrid' —— **推荐**。压缩包缓存在 Drive，每会话解压到本地盘。
#               一次下载永久有效；训练读取走本地盘全速。
#   'local'  —— 全在临时盘，用完即删，每个新会话都要重下。
DATA_MODE = 'hybrid'

# ── 数据规模 ───────────────────────────────────────────────────────────
# DNS read_speech 分片数。官方共 40 片、每片约 21 小时（299 GB / 40）。
# 取 1 片 ≈ 21 小时，落在"20~30 小时可管理子集"这个目标区间内。
# 分片名里的两个数字是 DNSMOS 区间（越靠后音质越好），这里取中段偏好的几片。
N_SPEECH_SHARDS = 1
# DNS 噪声分片：audioset（日常环境声）+ freesound（标注音效）各取几片。
N_AUDIOSET_SHARDS = 2
N_FREESOUND_SHARDS = 1

# ── 快速验证模式 ───────────────────────────────────────────────────────
# True = 跳过 DNS 大文件，只下 WenetSpeech（约 520 MB）验证整条链路。
QUICK_TEST = False

# ═══════════════════════════════════════════════════════════════════════

import os, sys, json, shutil, subprocess, time
from pathlib import Path
from shlex import quote as shq          # 路径里有空格时，所有 shell 命令都靠它

assert DATA_MODE in ('hybrid', 'local'), 'DATA_MODE 只能是 hybrid / local'

DRIVE = DRIVE_ROOT
WORK = WORK_ROOT
ARCHIVE_DIR = f'{WORK}/archives' if DATA_MODE == 'local' else f'{DRIVE}/archives'
DATA = f'{WORK}/data'
KEEP_ARCHIVE = DATA_MODE != 'local'

CKPT_DIR = f'{DRIVE}/checkpoints'   # 训练断点，每 epoch 保存
MODEL_DIR = f'{DRIVE}/models'       # 导出的 ONNX
TESTSET_DIR = f'{DRIVE}/testset'    # 固定测试集
LOG_DIR = f'{DRIVE}/logs'

assert os.path.isdir(DRIVE), (
    f'Drive 上找不到 {DRIVE}\n'
    '检查：① Drive 已挂载成功；② DRIVE_ROOT 与你实际的目录一致（区分大小写，空格照写）。'
)
for d in [WORK, ARCHIVE_DIR, DATA, CKPT_DIR, MODEL_DIR, TESTSET_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

print('目录布局')
print('─' * 74)
print(f'  代码包(需手动上传)  {DRIVE}/rtse-colab.zip')
print(f'  压缩包缓存          {ARCHIVE_DIR}')
print(f'  语料解压目标        {DATA}')
print(f'  数据清单            {DRIVE}/manifest.json')
print(f'  固定测试集          {TESTSET_DIR}')
print(f'  训练断点  ★         {CKPT_DIR}/<模型名>/{{last,best}}.pt')
print(f'  导出模型  ★         {MODEL_DIR}/<模型名>.onnx')
print('─' * 74)
print(f'  数据模式  {DATA_MODE}')
print(f'  语料      {"快速验证(仅 WenetSpeech)" if QUICK_TEST else "完整(DNS 英文训练 + WenetSpeech 中文评测)"}')
print()

!df -h /content | tail -1

In [ ]:
# ── 安装项目代码 ────────────────────────────────────────────────────────
# rtse-colab.zip 由本地 `uv run python scripts/pack_for_colab.py` 生成，
# 需要手动上传到 DRIVE_ROOT 目录下。**代码改过就要重新上传**，
# 否则 Colab 跑的还是旧逻辑（这个坑踩过，见 docs/ISSUES.md）。
ZIP = f'{DRIVE}/rtse-colab.zip'
assert os.path.exists(ZIP), (
    f'找不到 {ZIP}\n'
    '请先在本地执行 `uv run python scripts/pack_for_colab.py`，'
    f'再把 dist/rtse-colab.zip 上传到 Drive 的 {DRIVE} 下。\n'
    f'该目录下现有：{sorted(os.listdir(DRIVE))[:12]}'
)

SRC = f'{WORK}/rtse-src'
shutil.rmtree(SRC, ignore_errors=True)
os.makedirs(SRC, exist_ok=True)
!unzip -q -o {shq(ZIP)} -d {shq(SRC)}

# 只装项目需要而 Colab 没预装的。不用 `pip install -e .`：那会去解析 pyproject
# 里锁定的 torch CPU 索引，把 Colab 自带的 GPU 版 torch 覆盖掉，训练慢几十倍。
!pip install -q soxr pystoi jiwer webrtcvad-wheels pesq onnx onnxruntime opencc-python-reimplemented 2>&1 | tail -2

sys.path.insert(0, f'{SRC}/src')
import rtse
print('rtse', rtse.__version__, '| SR', rtse.SAMPLE_RATE, '| n_fft', rtse.N_FFT, '| hop', rtse.HOP_LENGTH)

import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# ── 自检：Colab 侧与本地必须是同一条信号链路 ───────────────────────────
# 这一步不能跳。Colab 上的 STFT 与本地哪怕差一点，训练出来的模型拿回本地就会
# 掉点，而且极难定位（两边单独看都"没问题"）。
import numpy as np, torch
from rtse.audio.stft import stft, istft, check_cola, magnitude_db
from rtse.data.dataset import stft_torch, istft_torch

x = np.random.default_rng(0).standard_normal(16000)
print('COLA 偏差          :', f'{check_cola():.2e}')
print('numpy 完美重构      :', f'{np.max(np.abs(istft(stft(x), length=x.size) - x)):.2e}')

xt = torch.from_numpy(x).float().unsqueeze(0)
ref, got = stft(x), stft_torch(xt)
assert ref.shape[0] == got.shape[2], f'帧数不一致 {ref.shape[0]} vs {got.shape[2]}'
gc = got[0,0].numpy() + 1j*got[0,1].numpy()
print('torch/numpy STFT   :', f'{np.max(np.abs(gc - ref)) / np.max(np.abs(ref)):.2e} (相对)')
print('torch 往返重构      :', f'{(istft_torch(stft_torch(xt), length=16000) - xt).abs().max().item():.2e}')

t = np.arange(16000)/16000
db = magnitude_db(stft(np.sin(2*np.pi*1000*t))).max()
print('dBFS 标定(满幅正弦) :', f'{db:.3f} dB  (应为 0.000)')
assert abs(db) < 0.05, 'dBFS 标定不对，检查代码包是否为最新'
print('\n✅ Colab 与本地是同一条链路。')

## 1. 下载 DNS Challenge 语料

分片是**独立的** `.tar.bz2`，可以只下需要的几片 —— 这是能做小规模子集的前提。
（DNS5 的 clean speech 是 `split` 切片，必须全部下载才能拼接解压，用不了。）

下载支持断点续传（`wget -c`）；已下好的会跳过，Colab 断线重连后重跑不会从头再来。

In [ ]:
DNS_BASE = 'https://dns4public.blob.core.windows.net/dns4archive/datasets_fullband'

def fetch_dns(name, blob_path, expect_min_wavs=50):
    """下载 → 解压一个 DNS 分片。带下载/解压双标记，支持断点续传与跨会话复用。

    校验方式是"解压后递归扫到的 wav 数量"，而不是断言某个具体子目录名 ——
    官方文档没有给出每个分片解压后的确切目录结构，而本机磁盘装不下几 GB 分片
    来提前验证（见 docs/ENVIRONMENT.md）。只要文件确实解出来了就算成功，
    下游 scan() 本来就是递归扫描，不关心嵌套了几层。
    """
    dl_mark = f'{ARCHIVE_DIR}/.{name}.downloaded'
    ex_mark = f'{DATA}/.{name}.extracted'
    fname = blob_path.rsplit('/', 1)[-1]
    archive = f'{ARCHIVE_DIR}/{fname}'
    out_dir = f'{DATA}/{name}'
    os.makedirs(out_dir, exist_ok=True)

    def count_wavs():
        r = subprocess.run(f'find {shq(out_dir)} -name "*.wav" | wc -l',
                           shell=True, capture_output=True, text=True)
        return int((r.stdout or '0').strip() or 0)

    if os.path.exists(ex_mark) and count_wavs() >= expect_min_wavs:
        print(f'[skip  ] {name} 已解压（{count_wavs()} 个 wav）'); return True

    if os.path.exists(dl_mark) and os.path.exists(archive):
        print(f'[cached] {name} 压缩包已在 Drive ({os.path.getsize(archive)/1e9:.2f} GB)')
    else:
        url = f'{DNS_BASE}/{blob_path}'
        print(f'[get   ] {name} ← {url}')
        rc = os.system(f'wget -q --show-progress -c -T 60 -O {shq(archive)} {shq(url)}')
        if rc != 0 or not os.path.exists(archive) or os.path.getsize(archive) < 1e6:
            print(f'[FAIL  ] {name} 下载失败。去 https://github.com/microsoft/DNS-Challenge '
                  f'确认 blob 路径是否变了。')
            return False
        Path(dl_mark).touch()

    print(f'[unpack] {name}  ({os.path.getsize(archive)/1e9:.2f} GB) → {out_dir}')
    t = time.time()
    os.system(f'tar -xjf {shq(archive)} -C {shq(out_dir)}')
    n = count_wavs()
    if n < expect_min_wavs:
        print(f'[FAIL  ] 解压后只找到 {n} 个 wav，压缩包可能不完整。'
              f'删掉 {archive} 和 {dl_mark} 后重跑本 cell')
        return False

    if not KEEP_ARCHIVE:
        os.remove(archive); Path(dl_mark).unlink(missing_ok=True)
    Path(ex_mark).touch()
    print(f'[done  ] {name}   {n} 个 wav   解压耗时 {time.time()-t:.0f} 秒')
    return True


def dns_shards():
    """按配置生成要下载的分片清单 (名字, blob 路径)。"""
    sp = [(f'dns_speech_{i:03d}',
           f'clean_fullband/datasets_fullband.clean_fullband.read_speech_{i:03d}_'
           + ['0.00_3.75','3.75_3.88','3.88_3.96','3.96_4.02','4.02_4.06'][i] + '.tar.bz2')
          for i in range(min(N_SPEECH_SHARDS, 5))]
    nz = [(f'dns_noise_audioset_{i:03d}',
           f'noise_fullband/datasets_fullband.noise_fullband.audioset_{i:03d}.tar.bz2')
          for i in range(N_AUDIOSET_SHARDS)]
    nz += [(f'dns_noise_freesound_{i:03d}',
            f'noise_fullband/datasets_fullband.noise_fullband.freesound_{i:03d}.tar.bz2')
           for i in range(N_FREESOUND_SHARDS)]
    # ⚠️ IR 分片在 blob 根目录下，**没有** `impulse_responses/` 前缀
    # （语音和噪声分片才有目录前缀）。写错会 404 —— 本地 HEAD 请求实测确认过。
    ir = [('dns_ir', 'datasets_fullband.impulse_responses_000.tar.bz2')]
    return sp, nz, ir

sp_shards, nz_shards, ir_shards = dns_shards()
print(f'计划下载：语音 {len(sp_shards)} 片、噪声 {len(nz_shards)} 片、IR {len(ir_shards)} 片')
free = shutil.disk_usage(DATA).free / 1e9
print(f'解压卷可用 {free:.1f} GB（估计需要 40~60 GB）')

if QUICK_TEST:
    print('\n[QUICK_TEST] 跳过 DNS 下载，只用 WenetSpeech 验证链路')
    results = {}
else:
    t0 = time.time()
    results = {}
    for n, b in sp_shards: results[n] = fetch_dns(n, b, expect_min_wavs=500)
    for n, b in nz_shards: results[n] = fetch_dns(n, b, expect_min_wavs=100)
    for n, b in ir_shards: results[n] = fetch_dns(n, b, expect_min_wavs=50)
    print(f'\n总耗时 {(time.time()-t0)/60:.1f} 分钟   结果: {results}')
    assert all(results.values()), '有分片没就绪，看上面的 FAIL 信息'

!df -h {shq(DATA)} | tail -1

## 2. 下载 WenetSpeech 中文评测集

**为什么用 HuggingFace 镜像而不是官方渠道**：WenetSpeech 官方
（wenet.org.cn）需要填 Google 表单拿密码，没法在 notebook 里脚本化。
`lmms-lab/WenetSpeech` 这个镜像不需要登录也不需要 token，
`test_meeting` 的音频和中文转写都打包在一个 220 MB 的 parquet 里。

> ⚠️ 该镜像的 `test_net` 只有 124 个文件（残缺），**不要用**。
> 完整可用的是 `dev`（13825 条）和 `test_meeting`（8370 条）。
> 这里用 `test_meeting` —— 真实会议录音，对"远场"这个主题比朗读语料更贴题。

In [ ]:
import pandas as pd

WNS_URL = ('https://huggingface.co/datasets/lmms-lab/WenetSpeech/resolve/main/'
           'data/test_meeting-00000-of-00001.parquet')
wns_parquet = f'{ARCHIVE_DIR}/wenetspeech_test_meeting.parquet'

if not os.path.exists(wns_parquet):
    print(f'[get   ] WenetSpeech test_meeting ← {WNS_URL}')
    rc = os.system(f'wget -q --show-progress -c -T 60 -O {shq(wns_parquet)} {shq(WNS_URL)}')
    assert rc == 0 and os.path.getsize(wns_parquet) > 1e6, '下载失败，检查网络或镜像是否还在'
else:
    print(f'[cached] WenetSpeech parquet 已存在 ({os.path.getsize(wns_parquet)/1e6:.0f} MB)')

wns = pd.read_parquet(wns_parquet)
print(f'\n共 {len(wns)} 条')
print('列：', list(wns.columns))
print('\n第一条样例：')
row0 = wns.iloc[0]
for c in wns.columns:
    v = row0[c]
    print(f'  {c}: {str(v)[:80] if not isinstance(v, (dict, bytes)) else type(v).__name__}')

## 3. 建立文件清单 + 噪声按平稳性分类

语音按**说话人**划分（避免同一人同时出现在训练和测试，模型靠记音色作弊）。
噪声用 `rtse.dsp.stationarity` 逐个算平稳性并分成两组 —— 这是后面
"DSP 在稳态噪声上够用、在非稳态上失效"这条对照能不能立住的前提。

In [ ]:
import random
from rtse.dsp.stationarity import stationarity_features, DEFAULT_DR_THRESHOLD_DB
from rtse.audio.io import read_audio
from tqdm.auto import tqdm

def scan(root, exts=('.wav', '.flac')):
    root = Path(root)
    if not root.exists(): return []
    return sorted(str(p) for p in root.rglob('*') if p.suffix.lower() in exts)

speech_all, noise_all, rir_all = [], [], []
if not QUICK_TEST:
    for n, _ in sp_shards: speech_all += scan(f'{DATA}/{n}')
    for n, _ in nz_shards: noise_all += scan(f'{DATA}/{n}')
    for n, _ in ir_shards: rir_all += scan(f'{DATA}/{n}')

print(f'语音 {len(speech_all):>7} 条')
print(f'噪声 {len(noise_all):>7} 条')
print(f'RIR  {len(rir_all):>7} 条')
if not QUICK_TEST:
    assert speech_all and noise_all and rir_all, '有类别扫不到文件，检查上一步解压结果'

In [ ]:
# ── 噪声平稳性分类 ─────────────────────────────────────────────────────
# 全量算太慢（几万个文件），抽样一部分做分类；每个文件只读前 10 秒就够判断。
MAX_NOISE_TO_CLASSIFY = 4000
rnd = random.Random(42)
noise_pool = noise_all[:]
rnd.shuffle(noise_pool)
noise_pool = noise_pool[:MAX_NOISE_TO_CLASSIFY]

stationary, nonstationary, skipped = [], [], 0
for p in tqdm(noise_pool, desc='噪声平稳性分类'):
    try:
        y = read_audio(p)[:16000 * 10]
    except Exception:
        skipped += 1; continue
    f = stationarity_features(y)
    if f.n_frames < 32:           # 太短，无法可靠判断
        skipped += 1; continue
    (stationary if f.detrended_dynamic_range_db < DEFAULT_DR_THRESHOLD_DB
     else nonstationary).append(p)

print(f'\n稳态   {len(stationary):>5} 条')
print(f'非稳态 {len(nonstationary):>5} 条')
print(f'跳过   {skipped:>5} 条（太短或读取失败）')
print(f'\n门限 {DEFAULT_DR_THRESHOLD_DB} dB —— 这个值是在合成噪声上标定的，'
      f'真实录音分布更连续，如果两组比例悬殊（比如 9:1）就该调它。')
assert stationary and nonstationary, '有一组是空的，门限需要重新标定'

In [ ]:
# ── 划分 train/test ────────────────────────────────────────────────────
# 语音按说话人（DNS 文件名形如 <speaker>_<utt>.wav，取第一段作为说话人 id）
def speaker_of(p):
    return Path(p).stem.split('_')[0]

spk = sorted({speaker_of(p) for p in speech_all})
random.Random(20260807).shuffle(spk)
n_val = max(2, len(spk) // 10)
spk_val = set(spk[:n_val])
split = {'train': [], 'val': []}
for p in speech_all:
    split['val' if speaker_of(p) in spk_val else 'train'].append(p)
print(f'说话人 {len(spk)} 位 → train {len(spk)-n_val} / val {n_val}')
for k, v in split.items():
    print(f'  语音 {k:>5}: {len(v):>7} 条')

# 噪声与 RIR 也划分：测试用的必须是训练没见过的
def split_list(xs, frac=0.2, seed=42):
    xs = xs[:]; random.Random(seed).shuffle(xs)
    cut = max(1, int(len(xs) * frac))
    return xs[cut:], xs[:cut]           # (train, test)

st_train, st_test = split_list(stationary)
ns_train, ns_test = split_list(nonstationary)
rir_train, rir_test = split_list(rir_all)
print(f'  稳态噪声  train/test: {len(st_train)}/{len(st_test)}')
print(f'  非稳态噪声 train/test: {len(ns_train)}/{len(ns_test)}')
print(f'  真实 RIR  train/test: {len(rir_train)}/{len(rir_test)}')

manifest = {
    'version': 'dns_wenetspeech',
    'quick_test': QUICK_TEST,
    'data_dir': DATA,
    'speech': split,
    'noise_train': st_train + ns_train,
    'noise_test': st_test + ns_test,
    'noise_stationary_train': st_train, 'noise_stationary_test': st_test,
    'noise_nonstationary_train': ns_train, 'noise_nonstationary_test': ns_test,
    'rir_train': rir_train, 'rir_test': rir_test,
    'stationarity_threshold_db': DEFAULT_DR_THRESHOLD_DB,
}
Path(f'{DRIVE}/manifest.json').write_text(
    json.dumps(manifest, ensure_ascii=False), encoding='utf-8')
print(f'\n清单已写入 {DRIVE}/manifest.json')

## 4. 合成固定测试集（中文 WenetSpeech + DNS 噪声 + 双 RIR）

**测试集必须固定下来**（预先合成好存盘），不能在线随机生成 ——
否则每次评测的噪声段和 SNR 都不同，前后两次跑出来的 CER 没有可比性。
训练集则相反，必须在线随机混音以扩大有效数据量。

### 分层设计

| 维度 | 取值 | 用途 |
|---|---|---|
| SNR | −5 / 0 / 5 / 10 / 15 dB | −5~0 压力测试；5~10 典型远场办公；15 验证"无害性" |
| 噪声 | 稳态 / 非稳态 | DSP vs NN 的核心对照 |
| RIR | 合成（RT60 0.2/0.4/0.6/0.8）+ 真实 | 受控扫描 + 泛化检验 |

**SNR 阶梯的理由**：
- **−5 ~ 0 dB**：重度淹没，最容易暴露"过抑制"——听着安静了但 ASR 断崖式下跌；
- **5 ~ 10 dB**：日常办公/会议室远场的典型信噪比，工程落地价值最核心的区间；
- **15 dB**：轻噪场景，评估**无害性**——好的增强算法在信噪比已经不错时
  不该引入额外失真让 CER 反而变差。

**RT60 阶梯的理由**：0.2 s 小型吸音房间；0.4~0.5 s 标准会议室/办公室；
0.8 s 大型未做吸音的教室或高挑大堂。混响会造成时域拖尾（前一个音素的反射
覆盖后一个），破坏 ASR 的声学特征边界。

In [ ]:
import io
import numpy as np
import soundfile as sf
from rtse.audio.io import write_audio
from rtse.data.synth import mix_at_snr, apply_rir, make_rir
from rtse.dsp.rt60 import estimate_t60

SNRS = [-5, 0, 5, 10, 15]
RT60S = [0.2, 0.4, 0.6, 0.8]
PER_CELL = 15
MIN_SEG_SEC, MAX_SEG_SEC = 3.0, 15.0

def decode_wns(rec):
    """从 parquet 一行里取出 (16kHz 波形, 中文转写)。

    HF 的音频列是 {'bytes': ..., 'path': ...}，用 soundfile 从内存解码
    （opus in ogg，libsndfile 1.2+ 支持），再重采样到项目统一的 16 kHz。
    """
    a = rec['audio']
    data, sr = sf.read(io.BytesIO(a['bytes']), dtype='float64', always_2d=False)
    if data.ndim == 2:
        data = data.mean(axis=1)
    if sr != 16000:
        import soxr
        data = soxr.resample(data, sr, 16000, quality='VHQ')
    txt = rec.get('text') or rec.get('sentence') or rec.get('transcription') or ''
    return data, str(txt)

# 挑长度合适、有转写的样本
cands = []
for i in range(len(wns)):
    r = wns.iloc[i]
    try:
        y, t = decode_wns(r)
    except Exception:
        continue
    d = y.size / 16000
    if MIN_SEG_SEC <= d <= MAX_SEG_SEC and len(t.strip()) >= 5:
        cands.append((y, t.strip()))
    if len(cands) >= 400:
        break
print(f'可用 WenetSpeech 样本 {len(cands)} 条')
assert cands, 'WenetSpeech 没解出可用样本，检查 parquet 的列名'
print('样例转写:', cands[0][1][:40])
print('样例时长: %.2f 秒' % (cands[0][0].size/16000))

In [ ]:
# 实验格：SNR × 噪声平稳性 × RIR 条件
rir_conditions = [('synth', t) for t in RT60S] + [('real', None)]
cells = [{'snr': s, 'noise_kind': nk, 'rir_kind': rk, 'rt60': rt}
         for s in SNRS for nk in ['stationary', 'nonstationary']
         for rk, rt in rir_conditions]
print(f'{len(cells)} 格 × {PER_CELL} 条 = {len(cells)*PER_CELL} 个样本')
print(f'  SNR {len(SNRS)} × 噪声 2 × RIR {len(rir_conditions)}（合成 {len(RT60S)} 档 + 真实 1 档）')

os.makedirs(f'{TESTSET_DIR}/audio', exist_ok=True)
# 先清空，保证目录内容严格等于 index.json —— 改小 PER_CELL 重跑时，
# 上一轮遗留的文件不会被覆盖，会变成索引里没有的孤儿文件混进 zip。
shutil.rmtree(f'{TESTSET_DIR}/audio', ignore_errors=True)
os.makedirs(f'{TESTSET_DIR}/audio', exist_ok=True)

rng = np.random.default_rng(20260807)
noise_by_kind = {'stationary': st_test, 'nonstationary': ns_test}
records, sidx = [], 0

def take_noise(paths, n, rng_):
    """从给定噪声列表取一段，裁/拼到 n 个样本。"""
    for _ in range(20):
        y = read_audio(paths[rng_.integers(len(paths))])
        if y.size >= 8000:
            break
    if y.size < n:
        y = np.tile(y, int(np.ceil(n / max(y.size, 1))))
    s = rng_.integers(0, y.size - n + 1) if y.size > n else 0
    return y[s:s+n]

for ci, cell in enumerate(tqdm(cells, desc='合成测试集')):
    for k in range(PER_CELL):
        clean, text = cands[(ci * PER_CELL + k) % len(cands)]
        n = clean.size
        clean = clean / (np.max(np.abs(clean)) + 1e-9) * 0.7

        if cell['rir_kind'] == 'synth':
            rir = make_rir(cell['rt60'], rng=rng)
            rt60_actual = round(float(estimate_t60(rir)), 3)
        else:
            rir = read_audio(rir_test[rng.integers(len(rir_test))])
            rt60_actual = round(float(estimate_t60(rir)), 3)
        wet = apply_rir(clean, rir)

        noise = take_noise(noise_by_kind[cell['noise_kind']], n, rng)
        noisy, _ = mix_at_snr(wet, noise, cell['snr'], rng=rng)

        stem = f"{sidx:05d}_{cell['noise_kind']}_snr{cell['snr']}_{cell['rir_kind']}"
        write_audio(f'{TESTSET_DIR}/audio/{stem}_noisy.wav', noisy)
        # 参考 = **混响后**的干净语音，不是原始干信号。否则降噪模型会因为
        # "没能去掉混响"被扣分，把降噪和去混响两件事混在一起。
        write_audio(f'{TESTSET_DIR}/audio/{stem}_clean.wav', wet)
        records.append({
            'id': stem, 'noisy': f'audio/{stem}_noisy.wav', 'clean': f'audio/{stem}_clean.wav',
            'text': text, 'duration_s': round(n/16000, 2),
            'snr': cell['snr'], 'noise_kind': cell['noise_kind'],
            'rir_kind': cell['rir_kind'],
            'rt60_nominal': cell['rt60'], 'rt60_measured': rt60_actual,
        })
        sidx += 1

Path(f'{TESTSET_DIR}/index.json').write_text(json.dumps({
    'sample_rate': 16000, 'version': 'dns_wenetspeech', 'per_cell': PER_CELL,
    'snrs': SNRS, 'rt60s': RT60S, 'records': records,
}, ensure_ascii=False, indent=1), encoding='utf-8')
print(f'\n测试集 {len(records)} 个样本 → {TESTSET_DIR}')
!du -sh "{TESTSET_DIR}"

### 校验

两件事必须确认，否则后面算出来的 CER 不可信：

1. **文本与音频对得上**：WenetSpeech 的音频是完整句子，没有做任何截断
   （`MIN_SEG_SEC`/`MAX_SEG_SEC` 是**筛选**条件，不是裁剪）——
   这一点靠代码结构保证，这里断言一下。
2. **合成 RIR 的实际 RT60 与标称值相符**：`make_rir` 的镜像阶数曾经写死，
   导致 RT60 超过 0.6 s 后完全失效（见 `docs/ISSUES.md` I-22）。
   已修复，但每次生成数据都值得复核一遍。

In [ ]:
import statistics as st

# 1) 音频没有被截断：所有时长都在筛选区间内，且不存在"卡在上限"的聚集
durs = [r['duration_s'] for r in records]
at_max = sum(1 for d in durs if abs(d - MAX_SEG_SEC) < 0.02)
print(f'时长 {min(durs):.2f}~{max(durs):.2f} 秒，均值 {st.mean(durs):.2f}')
print(f'恰好卡在上限 {MAX_SEG_SEC}s 的样本: {at_max} 条（大量聚集才说明被裁过）')
assert at_max < len(records) * 0.05, '大量样本卡在时长上限，检查是否误加了裁剪'

# 2) 合成 RIR 的实际 RT60 vs 标称
print('\n合成 RIR 的 RT60 标称 vs 实测:')
for t in RT60S:
    ms = [r['rt60_measured'] for r in records if r['rir_kind']=='synth' and r['rt60_nominal']==t]
    if ms:
        print(f'  标称 {t}s → 实测均值 {st.mean(ms):.3f}s')
real_rt = [r['rt60_measured'] for r in records if r['rir_kind']=='real'
           and r['rt60_measured'] == r['rt60_measured']]
if real_rt:
    print(f'\n真实 RIR 实测 RT60: {min(real_rt):.2f}~{max(real_rt):.2f}s，'
          f'中位数 {st.median(real_rt):.2f}s')

# 3) 分层是否齐全
import collections
print('\n各层样本数:', dict(collections.Counter(
    (r['noise_kind'], r['rir_kind']) for r in records)))
print('\n校验通过。')

## 5. 打包测试集，下载到本地

In [ ]:
!cd "{DRIVE}" && rm -f testset.zip && zip -q -r testset.zip testset && ls -lh testset.zip
print()
print('下一步：')
print('  1. 继续跑 02_train.ipynb（数据清单已就绪）')
print(f'  2. 从 Drive 下载 {DRIVE}/testset.zip，解压到本地项目的 data/ 下，')
print('     使 data/testset/index.json 存在，然后本地 `uv run rtse-eval` 就能跑')